In [1]:
import pandas as pd 
import numpy as np 
from faker import Faker
from sqlalchemy import create_engine

In [3]:
# Initialize Faker and set seed for reproducibility 
fake = Faker('en_IN') # Using Indian locale for realistic names/cities
np.random.seed(42)

# --- DATABASE CONFIGURATION --- 
DB_USER = 'root'
DB_PASSWORD = 'dhyanirohit%401234'
DB_HOST = 'localhost'
DB_NAME = 'credit_risk_db'


In [11]:
print("⏳ Generating Financial Data")

# --- 1. GENERATE CUSTOMER PROFILE DATA --- 
num_records = 5000
customer_ids = range(10000, 10000 + num_records)

customer_df = pd.DataFrame({
    'customer_id':customer_ids,
    'full_name': [fake.name() for _ in range(num_records)],
    'age' : np.random.randint(21,60, num_records),
    'city': [fake.city() for _ in range(num_records)], 
    'employement_years': np.random.randint(0, 35, num_records),
    'annual_income' : np.random.randint(30000, 3500000, num_records) # INR 3L to 35L
})

# --- 2. GENERATE LOAN & CREDIT DATA (WITH BUSINESS LOGIC) ---
# We calculate features that directly impact Credit Risk 

#  Basic Random Metrics 
credit_limits = np.random.randint(50000, 1000000, num_records)
outstanding_balances = credit_limits * np.random.uniform(0.1, 0.95, num_records) 
loan_amounts = np.random.randint(100000, 2500000, num_records)
existing_monthly_emi = np.random.randint(5000, 50000, num_records)

# Feature Engineering: Credit Utilization & DTI (Debt-to-Income) 
credit_utilization = outstanding_balances / credit_limits
monthly_income = customer_df['annual_income'] / 12
dti_ratio = existing_monthly_emi / monthly_income

# Simulating the Default Flag (The Target Variable)
# Rule: High DTI + High Utilization = Higher chance of default
default_probability = (credit_utilization * 0.4) + (dti_ratio * 0.4)  + np.random.uniform(0, 0.2, num_records)
# Threshold for default (top 15-20% riskiest profiles)
is_default = (default_probability > np.percentile(default_probability, 82)).astype(int) 

loan_df = pd.DataFrame({
    'loan_id' : range(50000, 50000 + num_records),
    'customer_id' : customer_ids,
    'loan_amount' : loan_amounts,
    'credit_limit' : credit_limits, 
    'outstanding_balance' : np.round(outstanding_balances, 2),
    'existing_monthly_emi' : existing_monthly_emi,
    'credit_utilization' : np.round(credit_utilization, 2) ,
    'dti_ratio' : np.round(dti_ratio, 2), 
    'loan_default' : is_default  # 1 = Default, 0 = Performing
})

print(f"✅ Generated {num_records} records") 
print(f"📊 Overall Default Rate : {loan_df ['loan_default'].mean():.2f}")

⏳ Generating Financial Data
✅ Generated 5000 records
📊 Overall Default Rate : 0.18


In [12]:
# --- 3. ETL: PUSH TO MYSQL --- 
try:
    print('⏳ Connecting to MySQL and pushing data..')
    # Create SQLAlchemy engine 
    engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

    # Push data to SQL tables 
    customer_df.to_sql('customers', con=engine, if_exists='replace', index=False)
    loan_df.to_sql('loans_and_credit', con=engine, if_exists='replace', index=False) 

    print("🚀 SUCCESS: Data pipeline executed. Tables 'customers' and 'loans_and_credit' are now in MySQL!")
    
except exception as e:
    print(f"ERROR connecting to My SQL:{e}")
    print("Please check your MySQL credentials, ensure the server is running, and that the database 'credit_risk_db' exists.")

⏳ Connecting to MySQL and pushing data..
🚀 SUCCESS: Data pipeline executed. Tables 'customers' and 'loans_and_credit' are now in MySQL!
